### Getting the cameras intrinsic parameters

In [ ]:
import cv2
import cv2.aruco as aruco
import glob
import os
import numpy as np
import re 
import pandas as pd
import math

In [81]:
cv2.__version__

'4.12.0'

In [82]:
experiment_no = 306

In [83]:
vid_files = glob.glob(f"D:/sleap-3D/session_{experiment_no}/gily_center/calibration_images/*.mp4")

In [84]:
image_save_folder = f"D:/sleap-3D/session_{experiment_no}/images/"

#### Uncomment below to save images of calibration from the video recorded

In [ ]:
# # Getting the images from video
# frames = []
# # Open video capture (e.g., from webcam)
# frame_counter = 0
# for i in vid_files:
#     cap = cv2.VideoCapture(i)

#     if not cap.isOpened():
#         raise Exception("Error: Could not open video stream.")

#     counter = 0
    
#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             print("Error: Failed to grab frame.")
#             break
#         if counter == 19:
#             cv2.imwrite(image_save_folder+f"frame_{frame_counter}.jpg",frame)
#             counter=0
#         frame_counter+=1
#         counter+=1
#     frame_counter+=1000

        
#         # frames.append(frame)
            
      

#### Uncomment below to get the cameras intrinsic parameters from the images stored from the prev cell

In [ ]:
# # I edited this using the python version of the original documentation - https://docs.opencv.org/4.12.0/da/d13/tutorial_aruco_calibration.html

# # Define the aruco dictionary and charuco board
# dictionary = aruco.getPredefinedDictionary(aruco.DICT_5X5_1000)
# board = aruco.CharucoBoard((12, 9), 0.03, 0.022, dictionary)
# # dictionary = cv2.aruco.getPredefinedDictionary(ARUCO_DICT)
# # board = cv2.aruco.CharucoBoard((SQUARES_VERTICALLY, SQUARES_HORIZONTALLY), SQUARE_LENGTH, MARKER_LENGTH, dictionary)
# # params = cv2.aruco.DetectorParameters()
# # detector = cv2.aruco.ArucoDetector(dictionary, params)
# charucodetector = cv2.aruco.CharucoDetector(board)

# # Load PNG images from folder
# image_files = [os.path.join(image_save_folder, f) for f in os.listdir(image_save_folder) if f.endswith(".jpg")]
# image_files.sort()  # Ensure files are in order

# all_charuco_corners = []
# all_charuco_ids = []
# all_object_points = []
# all_image_points = []

# for image_file in image_files:
#     image = cv2.imread(image_file)
#     image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#     image_copy = image.copy()
#     # marker_corners, marker_ids, _ = detector.detectMarkers(image)
#     charuco_corners, charuco_ids, marker_corners, marker_ids = charucodetector.detectBoard(image)
    
#     # If at least one marker is detected
#     if len(charuco_corners)>3:
#         # cv2.aruco.drawDetectedMarkers(image_copy, marker_corners, marker_ids)
#         # charuco_retval, charuco_corners, charuco_ids = cv2.aruco.interpolateCornersCharuco(marker_corners, marker_ids, image, board)
#         # if charuco_retval:
#         object_points, image_points = board.matchImagePoints(charuco_corners, charuco_ids)

#         if len(image_points)==0 or len(object_points)==0:
#             print("Point matching failed, try again.")
#             continue

#         all_charuco_corners.append(charuco_corners)
#         all_charuco_ids.append(charuco_ids)
#         all_object_points.append(object_points)
#         all_image_points.append(image_points)
        

# # Calibrate camera
# retval, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(all_object_points,all_image_points, image.shape[:2],None, None)

# # Save calibration data
# np.save(f"D:/sleap-3D/session_{experiment_no}/camera_matrix.npy", camera_matrix)
# np.save(f"D:/sleap-3D/session_{experiment_no}/dist_coeffs.npy", dist_coeffs)

### Getting Rotation and translation of the camera wrt the real world

In [ ]:

# Functions to sort file names correctly
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [atoi(c) for c in re.split(r'(\d+)', text)]


samplerate = 125000
camera_calibration_exp = 306



# I used this link to figure out how to get real world coordinates of the camera - https://bitw1z.medium.com/bridging-dimensions-camera-calibration-for-2d-to-3d-mapping-3d2b0a060a6f
# This gives us rotation and translation based on our coordinate system


# 2D pixel coordinates - this is got by adding nodes and marking points in sleap for which I know the real world coordinates
# The points are in session 306 in sleap_3D for reference
# change the values and run it for any experiment

# real world mic positions

# lowered mic positions - latest config assumes mics are at the surface
mic_positions = np.array([[0.2019, 0.8636, 0.0762],  # units in m
                    [0.9679, 0.8636, 0.0762],
                    [1.1684, 0.7621 , 0.0762],
                    [1.1684, 0.3911 , 0.0762],
                    [1.1684, 0.1871, 0.0762],  # units in m
                    [0.7859 , 0.0, 0.0762],
                    [0.3839, 0.0, 0.0762],
                    [0.0, 0.1031, 0.0762],
                    [0.0, 0.3911, 0.0762],
                    [0.0, 0.6781 , 0.0762]])

In [ ]:
# ----------------------------------------------------------------------------------------------------------------
# Use this code to get the rot and translational matrices of the camera in real world

# get pixel positions by loading video into sleap and add nodes - then save to csv
dat = pd.read_csv(f"D:/sleap-3D/session_{camera_calibration_exp}/pixel_positions.csv")
test_2d_pts = pd.read_csv(f"D:/sleap-3D/session_{camera_calibration_exp}/test_positions.csv")

points_2D_test_all = np.array([
             [test_2d_pts['new_part.x'][0],test_2d_pts['new_part.y'][0]],
             [test_2d_pts['new_part_1.x'][0],test_2d_pts['new_part_1.y'][0]],
             [test_2d_pts['new_part_2.x'][0],test_2d_pts['new_part_2.y'][0]],
             [test_2d_pts['new_part_3.x'][1],test_2d_pts['new_part_3.y'][1]],
             [test_2d_pts['new_part_4.x'][1],test_2d_pts['new_part_4.y'][1]],
             [test_2d_pts['new_part_5.x'][1],test_2d_pts['new_part_5.y'][1]],
             [test_2d_pts['new_part_6.x'][2],test_2d_pts['new_part_6.y'][2]],
             [test_2d_pts['new_part_7.x'][2],test_2d_pts['new_part_7.y'][2]],
             [test_2d_pts['new_part_8.x'][2],test_2d_pts['new_part_8.y'][2]],
             [test_2d_pts['new_part_9.x'][3],test_2d_pts['new_part_9.y'][3]],
             [test_2d_pts['new_part_10.x'][3],test_2d_pts['new_part_10.y'][3]],
             [test_2d_pts['new_part_11.x'][3],test_2d_pts['new_part_11.y'][3]],
             [test_2d_pts['new_part_12.x'][4],test_2d_pts['new_part_12.y'][4]],
             [test_2d_pts['new_part_13.x'][4],test_2d_pts['new_part_13.y'][4]],
             [test_2d_pts['new_part_14.x'][4],test_2d_pts['new_part_14.y'][4]],
             [test_2d_pts['new_part_15.x'][5],test_2d_pts['new_part_15.y'][5]],
             [test_2d_pts['new_part_16.x'][5],test_2d_pts['new_part_16.y'][5]],
             [test_2d_pts['new_part_17.x'][5],test_2d_pts['new_part_17.y'][5]],
             [test_2d_pts['new_part_18.x'][6],test_2d_pts['new_part_18.y'][6]],
             [test_2d_pts['new_part_19.x'][6],test_2d_pts['new_part_19.y'][6]],
             [test_2d_pts['new_part_20.x'][6],test_2d_pts['new_part_20.y'][6]],
             [test_2d_pts['new_part_21.x'][7],test_2d_pts['new_part_21.y'][7]],
             [test_2d_pts['new_part_22.x'][7],test_2d_pts['new_part_22.y'][7]],
             [test_2d_pts['new_part_23.x'][7],test_2d_pts['new_part_23.y'][7]]
             ],dtype='double')

# points_2D_test = points_2D_test_all[1::2, :]
points_2D_train = points_2D_test_all


# I've commented out some points to verify
points_2D = np.array([[dat['x_0.y_0.x'][0],dat['x_0.y_0.y'][0]], 
             [dat['x_1.y_0.x'][0],dat['x_1.y_0.y'][0]], 
             [dat['x_1.y_1.x'][0],dat['x_1.y_1.y'][0]],
             [dat['x_0.y_1.x'][0],dat['x_0.y_1.y'][0]],
             [dat['mic_0.x'][0],dat['mic_0.y'][0]],
             [dat['mic_1.x'][0],dat['mic_1.y'][0]],
             [dat['mic_2.x'][0],dat['mic_2.y'][0]],
             [dat['mic_3.x'][0],dat['mic_3.y'][0]],
             [dat['mic_4.x'][0],dat['mic_4.y'][0]],
             [dat['mic_5.x'][0],dat['mic_5.y'][0]],
             [dat['mic_6.x'][0],dat['mic_6.y'][0]],
             [dat['mic_7.x'][0],dat['mic_7.y'][0]],
             [dat['mic_8.x'][0],dat['mic_8.y'][0]],
             [dat['mic_9.x'][0],dat['mic_9.y'][0]]
             ],dtype='double')


points_2D = np.concatenate([points_2D,points_2D_train],axis=0)

points_3D_test_all = np.array([
            [0.400,0.5636, 0.0], # 0
            [0.000,0.5636,0.0], # 1
            [0.400,0.8636,0.0], # 2
            [0.300,0.4636 ,0.0], # 3
            [0.0,0.4636,0.0], # 4
            [0.300,0.8636,0.0], # 5
            [0.300,0.400,0.0], # 6
            [0.0,0.400,0.0], # 7
            [0.300,0.0,0.0], # 8
            [0.400,0.300,0.0], # 9
            [0.0,0.300,0.0], # 10
            [0.400,0.0,0.0], # 11
            [0.7684,0.300,0], # 12
            [1.1684,0.300,0.0], # 13
            [0.7684,0.0,0.0 ], # 14
            [0.8684,0.400,0.0], # 15
            [1.1684,0.400,0.0], # 16
            [0.8684,0.0,0.0], # 17
            [0.8684,0.4636,0.0], # 18
            [1.1684,0.4636,0.0], # 19
            [0.8684,0.8636,0.0], # 20
            [0.7684,0.5636,0.0], # 21
            [1.1684,0.5636,0.0], # 22
            [0.7684,0.8636,0.0] # 23
            ],dtype='double')

# points_3D_test = points_3D_test_all[1::2, :]
points_3D_train = points_3D_test_all

points_3D = np.array([[0.0,0.0,0.0],
            [1.1684,0.0,0.0],
            [1.1684,0.8636,0.0],
            [0.0,0.8636,0.0],
            mic_positions[0],
            mic_positions[1],
            mic_positions[2],
            mic_positions[3],
            mic_positions[4],
            mic_positions[5],
            mic_positions[6],
            mic_positions[7],
            mic_positions[8],
            mic_positions[9]
            ],dtype='double')


points_3D = np.concatenate([points_3D,points_3D_train],axis=0)


cameraMatrix = np.load(f"D:/sleap-3D/session_{camera_calibration_exp}/camera_matrix.npy")
distCoeffs = np.load(f"D:/sleap-3D/session_{camera_calibration_exp}/dist_coeffs.npy")

#solvePnp - to get 
print("Starting calibration")
retval, rvec, tvec = cv2.solvePnP(points_3D, points_2D, cameraMatrix, 
                                  distCoeffs, rvec=None, tvec=None, 
                                  useExtrinsicGuess=None, flags=None)

 

if retval: 
	rvec, _ = cv2.Rodrigues(rvec) # rotation matrix only
	np.save(f"D:/sleap-3D/session_{camera_calibration_exp}/cam_rotation.npy", rvec)
	np.save(f"D:/sleap-3D/session_{camera_calibration_exp}/cam_translation.npy", tvec)
print("Done with calibration")


#### Testing calibration error

In [ ]:
# -----------------------------------------------------------------------------------------------------------------------------

# loading the required values
# focal length and center pixels from intrisic parameters 
fx = cameraMatrix[0][0]
fy = cameraMatrix[1][1]
cx = cameraMatrix[0][2]
cy = cameraMatrix[1][2]



# rotation and translation matrix from world to camera coordinate system
rvec = np.load(f"D:/sleap-3D/session_{camera_calibration_exp}/cam_rotation.npy")
tvec = np.load(f"D:/sleap-3D/session_{camera_calibration_exp}/cam_translation.npy")

# rotation and translation matrix from world to camera coordinate system
R_cw = np.load(f"D:/sleap-3D/session_{camera_calibration_exp}/cam_rotation.npy")
t_cw = np.load(f"D:/sleap-3D/session_{camera_calibration_exp}/cam_translation.npy")


# given pixel values, convert it to real world coordinate system 
# def convert_2D_to_3D(u, v):
#     x = (u - cx) / fx
#     y = (v - cy) / fy
#     P_c = np.array([[x, y, 1]]).T
#     O_c = np.array([[0, 0, 0]]).T

#     R_wc = R_cw.T
#     P_w = np.dot(R_wc, (P_c - t_cw))
#     O_w = np.dot(R_wc, (O_c - t_cw))

#     line = P_w - O_w
#     line_z = line[2][0]
#     O_w_z = O_w[2][0]

#     k = (-1)*(O_w_z/line_z) # 0 = O_w_z + k*line_z
#     P = O_w + k*(P_w - O_w)
#     return P

# -----------------------------------------------------------------------------------------------------------------------
# testing calibration on some points

for _2d,_3d in zip(points_2D_test,points_3D_test): # need to include some test points from the pixel positions got
    # print(convert_2D_to_3D(_2d[0],_2d[1]),_3d)
    # print(math.dist(convert_2D_to_3D(_2d[0],_2d[1])[:2],_3d.flatten()[:2]))
    _2d_pred, _ = cv2.projectPoints(_3d,
                                 rvec, tvec,
                                 cameraMatrix,
                                 distCoeffs)
    print(_2d,_2d_pred)
    print(math.dist(_2d,_2d_pred.flatten()))

# ------------------------------------------------------------------------------------------------------------------------